# ***Statistical Learning on CASdatasets-$\texttt{fremotorclaim}$***

## ***Modeling Claim Count $N$***
In the following `code` cells, we compress the data frame, leaving only the explanatory risk factors and total claim counts for each risk profile. We then fit the Poisson GLM with and without `year` as a risk factor. The **likelihood-ratio test** finds that `year` is a statistically significant risk factor ($\Lambda = 353.4$, $q=2$, $p \approx 1.8\times10^{-77}$). 

Further, we compute the rate ratio of expected claim counts across years, holding other risk factors fixed. With `year = 7` as the baseline, years 8 and 9 have expected claim counts approximately 1.039 and 1.085 times that of year 7, respectively.

Finally, we split the data frame into training (years 7–8) and testing (year 9) datasets, with `year` encoded as numeric so the fitted  trend can be extrapolated to `year = 9`.

In [8]:
load("euMTPL.rda")
df_eu = euMTPL
df_eu$year = factor(df_eu$year)
df_eu$num_cl = df_eu$num_nc + df_eu$num_cg + df_eu$num_fcg + df_eu$num_cd # These are counts with different settlement methods
df_eu = df_eu[, c(1,3,4,5,6,8,9,10,11,20)]

In [9]:
dim(df_eu); head(df_eu); summary(df_eu)

[1] 2373197      10

,policy_id,fuel_type,year,vehicle_category,vehicle_use,horsepower,gender,age,exposure,num_cl
,<int>,<fct>,<fct>,<fct>,<fct>,<int>,<fct>,<int>,<dbl>,<int>
1,1,B,7,1,1,14,M,77,0.48767123,0
2,2,B,7,1,1,12,M,40,0.01917808,0
3,4,B,7,1,1,14,M,75,0.03287671,0
4,5,B,7,1,1,13,M,48,0.04383562,0
5,6,B,7,1,1,12,F,54,0.04657534,0
6,8,B,7,1,1,12,F,34,0.07671233,0


   policy_id         fuel_type       year       vehicle_category vehicle_use 
 Min.   :      1   B      :1382612   7:788932   1:2360481        0 :  12670  
 1st Qu.: 641820   D      : 531255   8:789367   8:  12716        1 :2359462  
 Median :1280437   G      : 383832   9:794898                    4 :    145  
 Mean   :1290070   T      :  29637                               5 :    518  
 3rd Qu.:1939574   P      :  19289                               26:    402  
 Max.   :2595214   S      :  15131                                           
                   (Other):  11441                                           
   horsepower     gender           age            exposure     
 Min.   :  0.00   F: 893348   Min.   : 18.00   Min.   :0.0010  
 1st Qu.: 14.00   M:1479849   1st Qu.: 37.00   1st Qu.:0.3005  
 Median : 16.00               Median : 46.00   Median :0.6822  
 Mean   : 16.52               Mean   : 48.17   Mean   :0.6266  
 3rd Qu.: 19.00               3rd Qu.: 59.00   3rd Qu.:1

In [10]:
# poisson.fit.full = glm(num_cl ~ fuel_type + year + vehicle_category + vehicle_use + horsepower + gender + age + offset(log(exposure)), family = poisson, df_eu)
# poisson.fit.reduced = glm(num_cl ~ fuel_type + vehicle_category + vehicle_use + horsepower + gender + age + offset(log(exposure)), family = poisson, df_eu)

In [11]:
# saveRDS(poisson.fit.full, "fit.full.rds")
# saveRDS(poisson.fit.reduced, "fit.reduced.rds")
# file.exists("fit.full.rds")
# file.exists("fit.reduced.rds")

In [12]:
fit.full = readRDS("fit.full.rds")
fit.reduced = readRDS("fit.reduced.rds")

In [13]:
anova(fit.full, fit.reduced, test = "LRT") 

,Resid. Df,Resid. Dev,Df,Deviance,Pr(>Chi)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,2373179,1624298,NA,NA,NA
2,2373181,1624652,-2,-353.4045,1.816285e-77


In [14]:
coefs = summary(fit.full)$coefficients
year_coefs = coefs[grep("year", rownames(coefs)), ]
contrasts(df_eu$year); exp(year_coefs[, "Estimate"]) #contrasts() shows that year 7 is the baseline.

,8,9
7,0,0
8,1,0
9,0,1


year8    year9 
1.038986 1.084528

In [ ]:
df_train = df_eu[df_eu$year %in% c("7", "8"), ]
df_test = df_eu[df_eu$year == "9", ]

df_train$year = as.numeric(as.character(df_train$year))
df_test$year = as.numeric(as.character(df_test$year))

fit.count.train = glm(num_cl ~ fuel_type + year + vehicle_category + vehicle_use + horsepower + gender + age + offset(log(exposure)), family = poisson, df_train)
pred.count.test = predict(fit.count.train, newdata = df_test, type = "response")

In [24]:
summary(df_eu)

   policy_id         fuel_type       year       vehicle_category vehicle_use 
 Min.   :      1   B      :1382612   7:788932   1:2360481        0 :  12670  
 1st Qu.: 641820   D      : 531255   8:789367   8:  12716        1 :2359462  
 Median :1280437   G      : 383832   9:794898                    4 :    145  
 Mean   :1290070   T      :  29637                               5 :    518  
 3rd Qu.:1939574   P      :  19289                               26:    402  
 Max.   :2595214   S      :  15131                                           
                   (Other):  11441                                           
   horsepower     gender           age            exposure     
 Min.   :  0.00   F: 893348   Min.   : 18.00   Min.   :0.0010  
 1st Qu.: 14.00   M:1479849   1st Qu.: 37.00   1st Qu.:0.3005  
 Median : 16.00               Median : 46.00   Median :0.6822  
 Mean   : 16.52               Mean   : 48.17   Mean   :0.6266  
 3rd Qu.: 19.00               3rd Qu.: 59.00   3rd Qu.:1

### ***Model Significance & Goodness-of-Fit Assessment***
After fitting the model, we would like to assess this model fit. The following `code` shows the $p$-value of the **Deviance statistic**, the **Likelihood-Ratio**, and **Pearson's chi-squared test**.

#### **Overall Model Significance (Likelihood-Ratio Test)**
We test whether the risk factors (`fuel_type`, `vehicle_category`, `vehicle_use`, `horsepower`, `gender`, `age`, and `year`) provide predictive power:
- **$H_0$:** $\boldsymbol{\beta} = \mathbf{0}$ (None of the risk factors has predictive power).
- **$H_1$:** At least one $\beta_j \neq 0$. 

$$\Delta D = D_{\text{Null}} - D_{\text{Res}} = 1,061,000 - 1,053,000 = 8,000$$

Under $H_0$, $\Delta D \sim \chi^2_{\Delta \text{df}}$, where $\Delta \text{df} = 1,578,298 - 1,578,282 = 16$. Computing $P(\chi^2_{16} \ge 8,000)$ yields $p \ll 2.2 \times 10^{-16}$. Thus, we **reject $H_0$**.

#### **Deviance Goodness-of-Fit Test**
- **$H_0$:** The Poisson GLM correctly describes the population.
- **$H_1$:** The Poisson GLM is misspecified.

Under $H_0$, the residual deviance follows $D_{\text{Res}} \sim \chi^2_{\text{df}_{\text{residual}}}$. We compute the upper-tail $p$-value:

$$p\text{-value} = P\left(\chi^2_{1,578,282} \ge 1,053,000\right) \approx 1.0$$

Since $p \ge 0.05$, we **fail to reject $H_0$**, indicating no evidence of structural underfit.

#### **Pearson's Chi-Squared Test**
We test whether the data exhibits overdispersion ($\text{Var}(Y) > \mu$), violating the standard Poisson variance assumption ($\phi = 1$):
- **$H_0$:** $\phi = 1$ (Equidispersion holds; variance equals the mean).
- **$H_1$:** $\phi > 1$ (The data is overdispersed).

We calculate the sum of squared Pearson residuals $X^2 = \sum_{i=1}^N e_i^2$, which, as a sum of squared standardized errors, follows a $\chi^2_{\text{df}_{\text{residual}}}$ distribution under $H_0$. Computing the upper-tail $p$-value yields $p \approx 0.0$.

---
***Final Verdict:*** The fitted Poisson GLM is fully well-specified. Both the mean and variance structures conform to model assumptions, meaning the estimated coefficient standard errors and $p$-values are reliable and require no Quasi-Poisson scaling or secondary dispersion adjustments.

In [27]:
p_val_lrt = pchisq(fit.count.train$null.deviance - fit.count.train$deviance, df = fit.count.train$df.null - fit.count.train$df.residual, lower.tail = FALSE)
p_val_gof = pchisq(fit.count.train$deviance, df = fit.count.train$df.residual, lower.tail = FALSE)
X2 = sum((pred.count.test-df_test$num_cl)^2/pred.count.test)
p = length(coef(fit.count.train))
df_test_residual = nrow(df_test) - p
p_val_X2 = pchisq(X2, df = df_test_residual, lower.tail = FALSE)

In [28]:
p_val_lrt; p_val_gof; p_val_X2

[1] 0

[1] 1

[1] 0

# 1. Initialize LFS in your repo
git lfs install

# 2. Tell LFS to track .rds and .tar.gz files
git lfs track "*.rds"
git lfs track "*.tar.gz"

# 3. Track the .gitattributes configuration file
git add .gitattributes